<center>
<img src="../../img/ods_stickers.jpg">
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
المؤلفون: [أولغا دايخوفسكايا](https://www.linkedin.com/in/odaykhovskaya/)، [يوري كاشنيتسكي](https://yorko.github.io). تخضع هذه المادة لشروط وأحكام ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). الاستخدام المجاني مسموح به لأي غرض غير تجاري.



# <center>المهمة رقم 7 (تجريبي). الحل
## <center> التعلم بدون إشراف
** نفس المهمة مثل [Kaggle Kernel](https://www.kaggle.com/kashnitsky/a7-demo-unsupervised-learning) + [الحل](https://www.kaggle.com/kashnitsky/a7-demo-unsupervised-learning-solution).**



في هذه المهمة، سننظر في كيفية عمل طرق تقليل أبعاد البيانات وتجميعها. وفي الوقت نفسه، سوف نتدرب على حل مهمة التصنيف مرة أخرى.
سنعمل مع مجموعة بيانات [Samsung Human Activity Recognition](https://archive.ics.uci.edu/ml/datasets/Human+Activity+Recognition+Using+Smartphones). قم بتنزيل البيانات [هنا](https://drive.google.com/file/d/14RukQ0ylM2GCdViUHBBjZ2imCaYcjlux/view?usp=sharing). تأتي البيانات من مقاييس التسارع والجيروسكوبات الخاصة بهواتف Samsung Galaxy S3 المحمولة (يمكنك العثور على مزيد من المعلومات حول الميزات باستخدام الرابط أعلاه)، كما يُعرف أيضًا نوع نشاط الشخص الذي يحمل هاتفًا في جيبه - سواء كان يمشي أو يقف أو يستلقي أو يجلس أو يصعد أو ينزل الدرج.
أولاً، نتظاهر بأن نوع النشاط غير معروف لنا، وسنحاول تجميع الأشخاص على أساس الميزات المتوفرة فقط. ومن ثم قمنا بحل مشكلة تحديد نوع النشاط البدني كمشكلة تصنيفية.
املأ الرمز عند الحاجة ("الرمز الخاص بك هنا") وأجب عن الأسئلة في [نموذج الويب](https://docs.google.com/forms/d/1wBf5UoRndv6PpzIwYnM9f0ysoGa4Yqcqle-HBlBP5QQ/edit).


In [ ]:
import os

import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm_notebook

%matplotlib inline
from matplotlib import pyplot as plt

plt.style.use(["seaborn-darkgrid"])
plt.rcParams["figure.figsize"] = (12, 9)
plt.rcParams["font.family"] = "DejaVu Sans"

from sklearn import metrics
from sklearn.cluster import AgglomerativeClustering, KMeans, SpectralClustering
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 17

In [ ]:
PATH_TO_SAMSUNG_DATA = "../../data/samsung_HAR"

In [ ]:
X_train = np.loadtxt(os.path.join(PATH_TO_SAMSUNG_DATA, "samsung_train.txt"))
y_train = np.loadtxt(
    os.path.join(PATH_TO_SAMSUNG_DATA, "samsung_train_labels.txt")
).astype(int)

X_test = np.loadtxt(os.path.join(PATH_TO_SAMSUNG_DATA, "samsung_test.txt"))
y_test = np.loadtxt(
    os.path.join(PATH_TO_SAMSUNG_DATA, "samsung_test_labels.txt")
).astype(int)

In [ ]:
# Checking dimensions
assert X_train.shape == (7352, 561) and y_train.shape == (7352,)
assert X_test.shape == (2947, 561) and y_test.shape == (2947,)


بالنسبة للتجميع، لا نحتاج إلى ناقل مستهدف، لذلك سنعمل مع مجموعة من عينات التدريب والاختبار. ادمج `X_train` مع `X_test`، و`y_train` مع `y_test`.


In [ ]:
# Your code here
X = np.vstack([X_train, X_test])
y = np.hstack([y_train, y_test])


تحديد عدد القيم الفريدة لتسميات الفئة المستهدفة.

In [ ]:
np.unique(y)

In [ ]:
n_classes = np.unique(y).size


[تتوافق هذه التصنيفات مع:](https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.names)
- 1 – المشي
- 2 - الصعود إلى الطابق العلوي
- 3 - النزول إلى الطابق السفلي
- 4 - الجلوس
- 5 - واقفاً
- 6 - الاستلقاء



قم بقياس العينة باستخدام `StandardScaler` مع المعلمات الافتراضية.


In [ ]:
# Your code here
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


قم بتقليل عدد الأبعاد باستخدام PCA، مع ترك أكبر عدد ممكن من المكونات اللازمة لشرح 90% على الأقل من تباين البيانات الأصلية (المقاسة). استخدم مجموعة البيانات المقاسة وأصلح `random_state` (ثابت RANDOM_STATE).


In [ ]:
# Your code here
pca = PCA(n_components=0.9, random_state=RANDOM_STATE).fit(X_scaled)
X_pca = pca.transform(X_scaled)


** السؤال 1: ** <br>
ما هو الحد الأدنى لعدد المكونات الرئيسية المطلوبة لتغطية 90% من تباين البيانات الأصلية (المقيسة)؟


In [ ]:
# В Your code here
X_pca.shape


**خيارات الإجابة:**
- 56 
- 65 **[+]**
- 66
- 193



** السؤال 2: **<br>
ما هي نسبة التباين التي يغطيها المكون الرئيسي الأول؟ التقريب إلى أقرب نسبة مئوية.
**خيارات الإجابة:**
- 45
- 51 **[+]**
- 56
- 61


In [ ]:
# Your code here
round(float(pca.explained_variance_ratio_[0] * 100))


تصور البيانات في الإسقاط على المكونين الرئيسيين الأولين.


In [ ]:
# Your code here
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, s=20, cmap="viridis");


**السؤال 3:**<br>
إذا سار كل شيء بشكل صحيح، فسترى عددًا من المجموعات، منفصلة تمامًا عن بعضها البعض تقريبًا. ما هي أنواع الأنشطة المدرجة في هذه المجموعات؟ <br>
**خيارات الإجابة:**
- مجموعة واحدة: جميع الأنشطة الستة
- مجموعتان: (المشي، الصعود، الصعود إلى الأسفل) و (الجلوس، الوقوف، الاستلقاء) ** [+] **
- 3 مجموعات: (المشي)، (المشي إلى الأعلى، المشي إلى الأسفل) و (الجلوس، الوقوف، الاستلقاء)
- 6 مجموعات



------------------------------


قم بإجراء التجميع باستخدام طريقة `KMeans`، وتدريب النموذج على البيانات ذات الأبعاد المنخفضة (بواسطة PCA). في هذه الحالة، سنقدم دليلًا للبحث عن 6 مجموعات بالضبط، لكن في الحالة العامة لن نعرف عدد المجموعات التي يجب أن نبحث عنها.
الخيارات:
- ** n_clusters ** = n_classes (عدد التصنيفات الفريدة للفئة المستهدفة)
- ** ن_ينيت ** = 100
- ** Random_state ** = RANDOM_STATE (لإعادة إنتاج النتيجة)
يجب أن يكون للمعلمات الأخرى قيم افتراضية.


In [ ]:
# Your code here
kmeans = KMeans(n_clusters=n_classes, n_init=100, random_state=RANDOM_STATE, n_jobs=1)
kmeans.fit(X_pca)
cluster_labels = kmeans.labels_


تصور البيانات في الإسقاط على المكونين الرئيسيين الأولين. تلوين النقاط حسب المجموعات التي تم الحصول عليها.


In [ ]:
# Your code here
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, s=20, cmap="viridis");


انظر إلى المراسلات بين علامات المجموعة وتسميات الفصل الأصلية وأنواع الأنشطة التي يتم الخلط بين خوارزمية `KMeans` فيها.


In [ ]:
tab = pd.crosstab(y, cluster_labels, margins=True)
tab.index = [
    "walking",
    "going up the stairs",
    "going down the stairs",
    "sitting",
    "standing",
    "laying",
    "all",
]
tab.columns = ["cluster" + str(i + 1) for i in range(6)] + ["all"]
tab


نرى أنه لكل فئة (أي كل نشاط) هناك عدة مجموعات. دعونا نلقي نظرة على الحد الأقصى للنسبة المئوية للكائنات الموجودة في الفصل والتي تم تعيينها لمجموعة واحدة. سيكون هذا مقياسًا بسيطًا يصف مدى سهولة فصل الفصل عن الآخرين عند التجميع.
مثال: إذا كان الفصل "يمشي في الطابق السفلي" (مع وجود 1406 نسخة تابعة له)، فإن توزيع المجموعات هو:
 - المجموعة 1 - 900
 - المجموعة 3 - 500
 - المجموعة 6 - 6،
 
فإن هذه المشاركة ستكون 900/1406 $ \approx $ 0.64.
 
** السؤال 4: ** <br>
ما هو النشاط الذي يتم فصله عن الباقي بشكل أفضل من الأنشطة الأخرى بناءً على المقياس البسيط الموضح أعلاه؟ <br>
**الجواب:**
- المشي
- واقفاً
- المشي في الطابق السفلي
- الخيارات الثلاثة كلها غير صحيحة** [+] **


In [ ]:
pd.Series(
    tab.iloc[:-1, :-1].max(axis=1).values / tab.iloc[:-1, -1].values,
    index=tab.index[:-1],
)

يمكن ملاحظة أن kMeans لا يميز بين الأنشطة بشكل جيد. استخدم طريقة الكوع لتحديد العدد الأمثل للمجموعات. معلمات الخوارزمية والبيانات التي نستخدمها هي نفسها كما كانت من قبل، ونغير فقط `n_clusters`.


In [ ]:
# Your code here
inertia = []
for k in tqdm_notebook(range(1, n_classes + 1)):
    kmeans = KMeans(n_clusters=k, n_init=100, random_state=RANDOM_STATE, n_jobs=1).fit(
        X_pca
    )
    inertia.append(np.sqrt(kmeans.inertia_))

In [ ]:
plt.plot(range(1, 7), inertia, marker="s");


نقوم بحساب $ D(k) $، كما هو موضح في مقالة [هذه](https://medium.com/open-machine-learning-course/open-machine-learning-course-topic-7-unsupervised-learning-pca-and-clustering-db7879568417) في قسم "اختيار عدد المجموعات لوسائل K".


In [ ]:
d = {}
for k in range(2, 6):
    i = k - 1
    d[k] = (inertia[i] - inertia[i + 1]) / (inertia[i - 1] - inertia[i])

In [ ]:
d


** السؤال 5: ** <br>
كم عدد العناقيد التي يمكننا اختيارها حسب طريقة الكوع؟ <br>
**خيارات الإجابة:**
- 1
- 2 **[+]**
- 3
- 4



------------------------



دعونا نجرب خوارزمية تجميع أخرى، موصوفة في المقالة – التجميع التكتل.


In [ ]:
ag = AgglomerativeClustering(n_clusters=n_classes, linkage="ward").fit(X_pca)


احسب مؤشر Rand المعدل (`sklearn.metrics`) للتجميع الناتج و` KMeans` باستخدام المعلمات من السؤال الرابع.


In [ ]:
# Your code here
print("KMeans: ARI =", metrics.adjusted_rand_score(y, cluster_labels))
print("Agglomerative CLustering: ARI =", metrics.adjusted_rand_score(y, ag.labels_))


** السؤال 6: ** <br>
حدد كافة العبارات الصحيحة. <br>
** خيارات الإجابة: **
- وفقًا لـ ARI، تعامل KMeans مع التجميع بشكل أسوأ من التجميع التجميعي ** [+] **
- بالنسبة لـ ARI، لا يهم العلامات التي تم تعيينها للمجموعة، فقط تقسيم المثيلات إلى مجموعات هو المهم ** [+] **
- في حالة التقسيم العشوائي إلى مجموعات، سيكون ARI قريباً من الصفر ** [+] **
**التعليق:**
1. نعم، كلما ارتفع معدل التهابات الجهاز التنفسي الحادة، كلما كان ذلك أفضل
2. نعم، إذا قمت بإعادة ترقيم المجموعات بشكل مختلف، فلن يتغير ARI
3. صحيح



-------------------------------



يمكنك ملاحظة أن المهمة لم يتم حلها بشكل جيد عندما نحاول اكتشاف عدة مجموعات (> 2). الآن، دعونا نحل مشكلة التصنيف، نظرًا لأن البيانات مصنفة.للتصنيف، استخدم جهاز ناقل الدعم – الفئة `sklearn.svm.LinearSVC`. في هذه الدورة قمنا بدراسة هذه الخوارزمية بشكل منفصل، ولكنها معروفة ويمكنك القراءة عنها، على سبيل المثال [هنا](http://cs231n.github.io/linear-classify/#svmvssoftmax).
اختر `C` المعلمة التشعبية لـ` LinearSVC` باستخدام `GridSearchCV`.
- تدريب `StandardScaler` الجديد على مجموعة التدريب (مع جميع الميزات الأصلية)، وتطبيق القياس على مجموعة الاختبار
- في `GridSearchCV`، حدد `cv` = 3.


In [ ]:
# Your code here
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
svc = LinearSVC(random_state=RANDOM_STATE)
svc_params = {"C": [0.001, 0.01, 0.1, 1, 10]}

In [ ]:
%%time
# Your code here
best_svc = GridSearchCV(svc, svc_params, n_jobs=1, cv=3, verbose=1)
best_svc.fit(X_train_scaled, y_train);

In [ ]:
best_svc.best_params_, best_svc.best_score_


**السؤال 7**<br>
ما هي قيمة المعلمة الفائقة `C` التي تم اختيارها الأفضل على أساس التحقق المتبادل؟ <br>
**خيارات الإجابة:**
- 0.001
- 0.01
- 0.1 **[+]**
- 1
- 10


In [ ]:
y_predicted = best_svc.predict(X_test_scaled)

In [ ]:
tab = pd.crosstab(y_test, y_predicted, margins=True)
tab.index = [
    "walking",
    "climbing up the stairs",
    "going down the stairs",
    "sitting",
    "standing",
    "laying",
    "all",
]
tab.columns = [
    "walking",
    "climbing up the stairs",
    "going down the stairs",
    "sitting",
    "standing",
    "laying",
    "all",
]
tab


كما ترون، تم حل مشكلة التصنيف بشكل جيد.



** السؤال 8: ** <br>
ما هو نوع النشاط الأسوأ اكتشافًا بواسطة SVM من حيث الدقة؟ هل تتذكر؟<br>
**خيارات الإجابة:**
- الدقة – صعود الدرج، التذكير – الاستلقاء
- الدقة – الاستلقاء – الاستذكار – الجلوس
- الدقة – المشي، التذكر – المشي
- الدقة - الوقوف، التذكر - الجلوس **[+]**
**التعليق:**
قام المصنف بحل المشكلة بشكل جيد، ولكن ليس بشكل مثالي.



أخيرًا، افعل نفس الشيء كما في السؤال 7، لكن أضف PCA.
- استخدم `X_train_scaled` و` X_test_scaled`
- قم بتدريب PCA نفسه كما كان من قبل، على مجموعة التدريب المتدرجة، وقم بتطبيق القياس على مجموعة الاختبار
- اختر المعلمة الفائقة `C` عبر التحقق المتبادل في مجموعة التدريب باستخدام تحويل PCA. ستلاحظ مدى سرعة عمله الآن.
** السؤال 9: ** <br>
ما هو الفرق بين أفضل جودة (دقة) للتحقق المتبادل في حالة جميع الخصائص الأولية البالغ عددها 561، وفي الحالة الثانية، عند تطبيق طريقة المكون الرئيسي؟ التقريب إلى أقرب نسبة مئوية. <br>** الخيارات: **
- الجودة هي نفسها
- 2%
- 4% **[+]**
- 10%
- 20%


In [ ]:
# Your code here
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=0.9, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [ ]:
svc = LinearSVC(random_state=RANDOM_STATE)
svc_params = {"C": [0.001, 0.01, 0.1, 1, 10]}

In [ ]:
%%time
best_svc_pca = GridSearchCV(svc, svc_params, n_jobs=1, cv=3, verbose=1)
best_svc_pca.fit(X_train_pca, y_train);

In [ ]:
best_svc_pca.best_params_, best_svc_pca.best_score_


وكانت النتيجة مع PCA أسوأ بنسبة 4%، مقارنة بالدقة عند التحقق المتبادل.


In [ ]:
round(100 * (best_svc_pca.best_score_ - best_svc.best_score_))


** السؤال 10: ** <br>
حدد جميع العبارات الصحيحة:
** خيارات الإجابة: **
- سمح تحليل المكون الرئيسي في هذه الحالة بتقليل وقت تدريب النموذج، في حين عانت الجودة (متوسط دقة التحقق المتبادل) بشكل كبير، بنسبة تزيد عن 10%
- يمكن استخدام PCA لتصور البيانات، ولكن هناك طرق أفضل لهذه المهمة، على سبيل المثال، tSNE. ومع ذلك، فإن PCA لديه تعقيد حسابي أقل ** [+] **
- يبني PCA مجموعات خطية من الميزات الأولية، وفي بعض التطبيقات قد يتم تفسيرها بشكل سيئ من قبل البشر ** [+] **
**التعليق:**
1. العبارة الأولى صحيحة، حيث سمح تحليل المكون الرئيسي في هذه الحالة بتقليل وقت تدريب النموذج بشكل كبير، ولكن الجودة لم تتأثر كثيرًا - بنسبة 4٪ فقط
2. لتصور البيانات متعددة الأبعاد، من الأفضل استخدام أساليب التعلم المتعددة، على وجه الخصوص، tSNE. في الوقت نفسه، لم يتم اختراع مقاييس تقييم جودة التصور بعد، ولكن tSNE يستخدم على نطاق واسع على وجه التحديد لأنه في بعض الحالات يبني صورًا "جيدة" توضح بنية البيانات، كما في حالة MNIST
3. غالبًا ما يتم تفسير المجموعات الخطية من الميزات، التي ينشئها PCA، بشكل سيئ من قبل البشر، على سبيل المثال، 0.574 \* الراتب + 0.234 \* num_children